## Pinecone Vector DB

In [1]:
# Create your index and apikey from here https://www.pinecone.io/
import os
api_key = os.getenv("PINECONE_API_KEY")

In [2]:
# !pip install -qU langchain langchain-pinecone langchain-openai

In [3]:
from pinecone import Pinecone
pc = Pinecone(api_key=api_key)

In [11]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small",
                              dimensions=1024,
                              api_key=os.getenv("OPENAI_API_KEY"))

In [12]:
from pinecone import ServerlessSpec

index_name = "rag-index"  # change if desired

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=1024,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

In [13]:
index

In [14]:
from langchain_pinecone import PineconeVectorStore

vector_store = PineconeVectorStore(index=index, embedding=embeddings)

In [15]:
vector_store

In [16]:
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]

In [17]:
vector_store.add_documents(documents=documents)

['c3756a70-8e65-41d5-b785-acb3ba69a0f6',
 'f91a3b45-481a-4ea7-b825-4a24cd6584ac',
 '62222e8d-9c65-4ab2-88c2-f73503874616',
 '6a53a29f-dd25-49d5-9625-e0cd696edf9a',
 '54cad32f-2284-4f3c-8c3a-c9214835453d',
 '48f579b9-f33a-4d2e-816c-eadebe4bbca7',
 'f9914139-dfbf-4f15-b5ee-cb686e5582fc',
 'f20e844d-1257-44da-a737-47e84158efc2',
 '9a8da02e-34db-4b4a-86cc-af65845b2e8e',
 'dbd3d30b-48a2-4369-b1c2-993d725a4a1e']

In [18]:
# Query Directly
results = vector_store.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    k=2,
    filter={"source": "tweet"},
)

for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

* Building an exciting new project with LangChain - come check it out! [{'source': 'tweet'}]
* LangGraph is the best framework for building stateful, agentic applications! [{'source': 'tweet'}]


In [19]:
results = vector_store.similarity_search_with_score(
    "Will it be hot tomorrow?", k=1, filter={"source": "news"}
)
for res, score in results:
    print(f"* [SIM={score:3f}] {res.page_content} [{res.metadata}]")

* [SIM=0.573013] The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees. [{'source': 'news'}]


In [20]:
# Retriever
retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 1, "score_threshold": 0.4},
)
retriever.invoke("Stealing from the bank is a crime", filter={"source": "news"})

[Document(id='6a53a29f-dd25-49d5-9625-e0cd696edf9a', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.')]